In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("OlistPipeline").getOrCreate()

In [3]:
spark

In [4]:
orders_spark = spark.read.csv("data/olist_orders_dataset.csv", header=True, inferSchema=True)
orders_spark.show(5)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [5]:
delivered_orders = orders_spark.filter(orders_spark.order_status == "delivered")

In [6]:
delivered_orders.explain()

== Physical Plan ==
*(1) Filter (isnotnull(order_status#19) AND (order_status#19 = delivered))
+- FileScan csv [order_id#17,customer_id#18,order_status#19,order_purchase_timestamp#20,order_approved_at#21,order_delivered_carrier_date#22,order_delivered_customer_date#23,order_estimated_delivery_date#24] Batched: false, DataFilters: [isnotnull(order_status#19), (order_status#19 = delivered)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/ilayda/olist-data-pipeline/data/olist_orders_dataset.csv], PartitionFilters: [], PushedFilters: [IsNotNull(order_status), EqualTo(order_status,delivered)], ReadSchema: struct<order_id:string,customer_id:string,order_status:string,order_purchase_timestamp:timestamp,...




In [7]:
delivered_orders.count()

96478

### Lazy Evaluation: Transformation vs Action

**Deney:** `orders_spark.filter(orders_spark.order_status == "delivered")` çalıştırıldığında hiçbir çıktı alınmadı — bu bir **transformation**, Spark sadece bir plan oluşturdu. `.explain()` ile bu plan (Physical Plan) görüntülendi; Spark'ın otomatik olarak `isnotnull` kontrolü eklediği ve **predicate pushdown** (filtreyi veri kaynağına yakın uygulama) yaptığı gözlemlendi. Ardından `.count()` çağrıldığında (bir **action**), Spark planı gerçekten çalıştırdı ve sonucu döndürdü: 96478 delivered sipariş.

**Pandas karşılığı:** `orders_df[orders_df["order_status"] == "delivered"]` yazıldığı an filtreleme **anında** gerçekleşir ve bellekte yeni bir DataFrame oluşur — Spark'taki gibi "plan yapıp bekleme" davranışı yoktur.

**Temel fark:** Pandas eager (istekli) çalışır, her satır yazıldığı anda işlenir. Spark lazy (tembel) çalışır, transformation'lar biriktirilir ve yalnızca bir action tetiklendiğinde optimize edilerek çalıştırılır.

In [8]:
order_items_spark = spark.read.csv("data/olist_order_items_dataset.csv", header=True, inferSchema=True)
products_spark = spark.read.csv("data/olist_products_dataset.csv", header=True, inferSchema=True)
order_items_spark.show(5)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18|12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51|199.9|        18.14|
+--------------------+-------------+------------

In [9]:
from pyspark.sql.functions import sum as spark_sum

kategori_gelir_spark = (
    order_items_spark
    .join(products_spark, on="product_id", how="inner")
    .groupBy("product_category_name")
    .agg(spark_sum("price").alias("toplam_gelir"))
    .orderBy(spark_sum("price").desc())
)

kategori_gelir_spark.show(10)

+---------------------+------------------+
|product_category_name|      toplam_gelir|
+---------------------+------------------+
|         beleza_saude|1258681.3399999938|
|   relogios_presentes| 1205005.679999998|
|      cama_mesa_banho|1036988.6799999807|
|        esporte_lazer| 988048.9699999837|
| informatica_acess...|  911954.319999988|
|     moveis_decoracao| 729762.4899999866|
|           cool_stuff| 635290.8499999974|
| utilidades_domest...| 632248.6599999928|
|           automotivo|  592720.109999997|
|   ferramentas_jardim|485256.45999999647|
+---------------------+------------------+
only showing top 10 rows


### Transformation 1: Kategoriye Göre Toplam Gelir (Filtreleme + Gruplama)

**Spark syntax'ı:** `order_items_spark.join(products_spark, on="product_id").groupBy("product_category_name").agg(spark_sum("price"))`

**Pandas karşılığı:** `order_items_df.merge(products_df, on="product_id").groupby("product_category_name")["price"].sum()`

**Yorum:** Syntax olarak iki kütüphane de oldukça benzer bir mantık izliyor (join/merge → groupby → agg). Temel fark, Spark'ın bu işlemi yazdığın anda çalıştırmaması (lazy evaluation) — `.show()` çağrılana kadar gerçek hesaplama yapılmıyor. Ayrıca Spark'ta agregasyon fonksiyonları (`sum`, `avg` vb.) Python'un yerleşik fonksiyonlarıyla karışmaması için ayrıca import edilmesi gerekiyor, Pandas'ta böyle bir gereklilik yok.

**Doğrulama notu:** Bu sonuçlar, Phase 2'de SQL ile bulunan kategori bazlı gelir analiziyle (health_beauty, watches_gifts, bed_bath_table sıralamasıyla) birebir örtüşüyor — sadece kategori isimleri burada Portekizce (translation JOIN'i yapılmadığı için).

In [10]:
from pyspark.sql.functions import datediff

orders_with_delivery = orders_spark.withColumn(
    "teslimat_suresi_gun",
    datediff(orders_spark.order_delivered_customer_date, orders_spark.order_purchase_timestamp)
)

orders_with_delivery.select("order_id", "order_purchase_timestamp", "order_delivered_customer_date", "teslimat_suresi_gun").show(5)

+--------------------+------------------------+-----------------------------+-------------------+
|            order_id|order_purchase_timestamp|order_delivered_customer_date|teslimat_suresi_gun|
+--------------------+------------------------+-----------------------------+-------------------+
|e481f51cbdc54678b...|     2017-10-02 10:56:33|          2017-10-10 21:25:13|                  8|
|53cdb2fc8bc7dce0b...|     2018-07-24 20:41:37|          2018-08-07 15:27:45|                 14|
|47770eb9100c2d0c4...|     2018-08-08 08:38:49|          2018-08-17 18:06:29|                  9|
|949d5b44dbf5de918...|     2017-11-18 19:28:06|          2017-12-02 00:28:42|                 14|
|ad21c59c0840e6cb8...|     2018-02-13 21:18:39|          2018-02-16 18:17:02|                  3|
+--------------------+------------------------+-----------------------------+-------------------+
only showing top 5 rows


### Transformation 2: Teslimat Süresi Hesaplama (Yeni Kolon Türetme)

**Spark syntax'ı:** `orders_spark.withColumn("teslimat_suresi_gun", datediff(orders_spark.order_delivered_customer_date, orders_spark.order_purchase_timestamp))`

**Pandas karşılığı:** `orders_df["teslimat_suresi_gun"] = (orders_df["order_delivered_customer_date"] - orders_df["order_purchase_timestamp"]).dt.days`

**Yorum:** Pandas'ta yeni bir kolon türetmek çok basit — doğrudan `df["yeni_kolon"] = ...` şeklinde atama yapılır. Spark'ta ise DataFrame'ler **immutable** (değiştirilemez) olduğu için `df["yeni_kolon"] = ...` gibi bir atama çalışmaz; bunun yerine `.withColumn(...)` metodu kullanılır, bu da yeni kolonu içeren **yeni bir DataFrame döndürür** (orijinal DataFrame değişmez). Ayrıca tarih farkı için Pandas'ta `.dt.days` kullanılırken, Spark'ta özel bir fonksiyon (`datediff`) import edilip kullanılması gerekiyor.

In [12]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

musteri_pencere = Window.partitionBy("customer_id").orderBy("order_purchase_timestamp")

orders_with_rank = orders_spark.withColumn(
    "musteri_siparis_no",
    row_number().over(musteri_pencere)
)

orders_with_rank.select("customer_id", "order_purchase_timestamp", "musteri_siparis_no").show(10)

+--------------------+------------------------+------------------+
|         customer_id|order_purchase_timestamp|musteri_siparis_no|
+--------------------+------------------------+------------------+
|00050bf6e01e69d5c...|     2017-09-17 16:04:44|                 1|
|000598caf2ef41174...|     2018-08-11 12:14:35|                 1|
|000bf8121c3412d30...|     2017-10-11 07:44:31|                 1|
|00114026c1b7b52ab...|     2017-06-01 19:44:44|                 1|
|0013cd8e350a7cc76...|     2018-05-07 23:25:09|                 1|
|0015bc9fd2d539544...|     2018-06-11 19:48:34|                 1|
|0015f7887e2fde13d...|     2017-07-31 11:05:46|                 1|
|001df1ee5c36767aa...|     2018-08-05 23:14:45|                 1|
|001f150aebb5d897f...|     2018-05-09 12:06:04|                 1|
|001f35d9f262c558f...|     2018-01-25 17:33:55|                 1|
+--------------------+------------------------+------------------+
only showing top 10 rows


In [13]:
orders_with_rank.filter(orders_with_rank.musteri_siparis_no > 1).show(5)

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|musteri_siparis_no|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+------------------+
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+------------------+



### Transformation 3: Müşteri Bazlı Sipariş Sıralaması (Window Function)

**Spark syntax'ı:** `Window.partitionBy("customer_id").orderBy("order_purchase_timestamp")` tanımlanıp `row_number().over(pencere)` ile her satıra kendi müşteri grubundaki sıra numarası verildi.

**Pandas karşılığı:** `orders_df.groupby("customer_id")["order_purchase_timestamp"].rank(method="first")` veya `orders_df.sort_values("order_purchase_timestamp").groupby("customer_id").cumcount() + 1`

**Yorum:** `GROUP BY` bir grubu tek satıra indirger, ama window function hiçbir satırı kaybetmeden her satıra "kendi grubundaki konumu" bilgisini ekler. Pandas'ta bu, `groupby()` + `cumcount()`/`rank()` kombinasyonuyla yapılabiliyor ama SQL/Spark'taki `Window` yapısı kadar esnek ve okunaklı değil.

**Bulgu:** `musteri_siparis_no > 1` filtresi **tamamen boş sonuç** döndürdü — yani veri setindeki hiçbir müşteri birden fazla sipariş vermemiş. Bu, platformda tekrar eden müşteri (repeat customer) oranının bu dönemde neredeyse sıfır olduğunu gösteriyor; büyüme büyük ölçüde yeni müşteri kazanımına dayanıyor olabilir. Bu, sonuç/öneri bölümünde müşteri sadakat programı önerisi için güçlü bir kanıt olarak kullanılabilir.

In [16]:
kategori_gelir_pandas = kategori_gelir_spark.toPandas()
kategori_gelir_pandas.to_parquet("outputs/kategori_gelir.parquet", index=False)

### Sonuçların Parquet Formatında Kaydedilmesi

**Neden Parquet, neden CSV değil:** Parquet, sütun bazlı (columnar) bir depolama formatıdır — CSV'nin aksine veri tiplerini içinde saklar, sıkıştırılmış haldedir ve analitik sorgularda (özellikle belirli kolonların okunduğu durumlarda) çok daha verimlidir.

**Teknik not:** Spark'ın kendi `.write.parquet(...)` metodu, Windows'ta Hadoop'un winutils.exe bağımlılığı nedeniyle hata verdi (`HADOOP_HOME and hadoop.home.dir are unset`) — bu, yalnızca Windows'ta yerel/local Spark kurulumuna özgü bir kısıtlama olup, gerçek bir cluster ortamında (Linux tabanlı production sistemlerde) karşılaşılmaz. Bunun yerine, küçük ve zaten toplanmış (aggregate edilmiş) olan sonuç `.toPandas()` ile Pandas DataFrame'ine çevrilip, `pyarrow` motoruyla Parquet olarak kaydedildi. Bu yaklaşım büyük ham veri setleri için uygun olmazdı (tüm veriyi tek makineye toplamak gerekirdi), ama küçük özet sonuçlar için pratik ve gerçek dünyada da sık kullanılan bir yaklaşımdır.

## Sonuç ve Özet Bulgular

Bu bölüm, EDA (Phase 1), SQL analizi (Phase 2) ve PySpark transformasyonlarından (Phase 3) elde edilen bulguları özetlemektedir.

### Gelir ve Ödeme
- Credit card, işlemlerin %74'ünü ve toplam gelirin %78'ini oluşturan baskın ödeme yöntemi; aynı zamanda en yüksek ortalama işlem tutarına (163.32) sahip.
- En yüksek gelirli kategoriler `health_beauty` ve `watches_gifts` — ikincisi, çok daha az satış adediyle neredeyse aynı geliri getiriyor, bu da daha yüksek birim fiyatlı bir kategori olduğuna işaret ediyor.

### Teslimat Performansı
- Brezilya'nın kuzey/Amazon bölgesindeki eyaletler (RR, AP, AM) en uzun ortalama teslimat sürelerine sahip (27-29 gün), ana lojistik merkezlerden coğrafi uzaklıkla örtüşüyor.
- Müşteri memnuniyet skoru ile teslimat süresi arasında güçlü bir ters ilişki var: 1 yıldızlı siparişlerin ortalama teslimat süresi (21.3 gün), 5 yıldızlıların (10.7 gün) neredeyse iki katı.

### Müşteri Davranışı
- Veri setindeki hiçbir müşteri birden fazla sipariş vermemiş — tekrar eden müşteri oranı pratik olarak %0.

### Veri Kalitesi Notları
- 2016-09/2016-10/2016-12 ve 2018-09 dönemlerinde anormal derecede düşük sipariş sayıları gözlemlendi — bu, gerçek bir iş sorunu değil, veri setinin bu tarihlerde henüz başlamamış/kesilmiş olmasından kaynaklanıyor.
- `payments` tablosunda "not_defined" olarak etiketlenmiş ve değeri 0 olan az sayıda (3) kayıt tespit edildi — küçük ölçekli ama not edilmesi gereken bir veri kalitesi sorunu.